# FD004 multi-prefix late-stage guard v02

Test simple late-stage guards on existing multi-prefix predictions without retraining or test-final usage.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "results" / "FD004" / "FD004_multiprefix_late_stage_guard_results_v02.csv").exists():
        PROJECT_ROOT = candidate
        break

results = pd.read_csv(PROJECT_ROOT / "results/FD004/FD004_multiprefix_late_stage_guard_results_v02.csv")
predictions = pd.read_csv(PROJECT_ROOT / "results/FD004/FD004_multiprefix_late_stage_guard_predictions_v02.csv")
best = pd.read_csv(PROJECT_ROOT / "results/FD004/FD004_multiprefix_late_stage_guard_best_candidate_v02.csv")
bins = pd.read_csv(PROJECT_ROOT / "results/FD004/FD004_multiprefix_late_stage_guard_metrics_by_bin_v02.csv")
diagnostics = pd.read_csv(PROJECT_ROOT / "results/FD004/FD004_multiprefix_late_stage_guard_diagnostics_v02.csv")

## Results


In [ ]:
display(results.sort_values("CMAPSS_mean"))
display(results[results["acceptable"]].sort_values(["CMAPSS_mean", "dangerous_20_pct", "MAE_RUL_le_50", "RMSE"]))
display(diagnostics)

best_name = best.loc[0, "candidate_name"]
base = predictions[predictions["candidate_name"].eq("baseline")].copy()
eol = predictions[predictions["candidate_name"].eq("eol_mean")].copy()
guard = predictions[predictions["candidate_name"].eq(best_name)].copy()

fig, ax = plt.subplots(figsize=(5, 5))
for label, frame in [("baseline", base), ("eol_mean", eol), (best_name, guard)]:
    ax.scatter(frame["true_RUL"], frame["pred_RUL"], s=12, alpha=0.45, label=label)
lim = max(predictions["true_RUL"].max(), predictions["pred_RUL"].max())
ax.plot([0, lim], [0, lim], color="black", linewidth=1)
ax.set_xlabel("True RUL")
ax.set_ylabel("Predicted RUL")
ax.set_title("True vs predicted RUL")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for label, frame in [("baseline", base), ("eol_mean", eol), (best_name, guard)]:
    ax.scatter(frame["true_RUL"], frame["error"], s=12, alpha=0.45, label=label)
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("True RUL")
ax.set_ylabel("Pred - true")
ax.set_title("Residuals by true RUL")
ax.legend()
plt.show()

plot_bins = bins[bins["candidate_name"].isin(["baseline", "eol_mean", best_name])].copy()
for value_col, title in [
    ("MAE", "MAE by RUL bin"),
    ("dangerous_20_pct", "Dangerous_20 by RUL bin"),
    ("CMAPSS_share", "CMAPSS share by RUL bin"),
]:
    pivot = plot_bins.pivot(index="rul_bin", columns="candidate_name", values=value_col)
    display(pivot)
    pivot.plot(kind="bar", figsize=(7, 4), title=title)
    plt.tight_layout()
    plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
guard["delta_vs_eol"].hist(ax=axes[0], bins=30)
axes[0].set_title(f"{best_name}: guard_pred - eol_mean_pred")
guard["delta_vs_baseline"].hist(ax=axes[1], bins=30)
axes[1].set_title(f"{best_name}: guard_pred - baseline_pred")
plt.tight_layout()
plt.show()

for label, frame in [("baseline", base), ("eol_mean", eol), (best_name, guard)]:
    display(Markdown(f"### Top 10 CMAPSS - {label}"))
    display(frame.sort_values("cmapss_penalty", ascending=False).head(10)[["unit_id", "split_id", "cycle", "true_RUL", "pred_RUL", "error", "cmapss_penalty", "guard_applied"]])

changed = guard[guard["guard_applied"]].sort_values("cycle").head(8)
display(Markdown("### Examples where selected guard changes prediction"))
display(changed[["unit_id", "split_id", "cycle", "true_RUL", "baseline_pred", "eol_mean_pred", "pred_RUL", "delta_vs_eol", "delta_vs_baseline"]])
for _, row in changed[["unit_id", "split_id"]].drop_duplicates().head(4).iterrows():
    mask = guard["unit_id"].eq(row["unit_id"]) & guard["split_id"].eq(row["split_id"])
    unit_guard = guard[mask].sort_values("cycle")
    unit_base = base[mask].sort_values("cycle")
    unit_eol = eol[mask].sort_values("cycle")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(unit_base["cycle"], unit_base["true_RUL"], marker="o", label="true RUL")
    ax.plot(unit_base["cycle"], unit_base["pred_RUL"], marker="o", label="baseline")
    ax.plot(unit_eol["cycle"], unit_eol["pred_RUL"], marker="o", label="eol_mean")
    ax.plot(unit_guard["cycle"], unit_guard["pred_RUL"], marker="o", label=best_name)
    ax.set_title(f"Unit {int(row['unit_id'])} split {int(row['split_id'])}")
    ax.set_xlabel("cycle")
    ax.set_ylabel("RUL")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Decision final v02

- Mejor candidato encontrado: `eol_mean_guard_b60_only_if_lower_floor2`.
- Comparacion contra baseline actual `fd004_high_rul_thr120_off2`: CMAPSS mean `4.9531` -> `3.0618`, RMSE `15.7785`, dangerous_20 `6.69%` -> `0.40%`, MAE_RUL_le_50 `7.7906` -> `5.8628`.
- Comparacion contra `eol_mean` sin guard: CMAPSS mean `3.2217` -> `3.0618`, dangerous_20 `0.40%` -> `0.40%`, MAE_RUL_le_50 `7.8523` -> `5.8628`.
- Recomendacion: adoptar `eol_mean_guard_b60_only_if_lower_floor2` para FD004 v02.
- Razon: Adopted: selected guard satisfies acceptance constraints with best CMAPSS priority.
- Nota: `eol_mean` sin guard ya cumple la tolerancia de MAE_RUL_le_50 (+0.10), pero el guard seleccionado mejora mas CMAPSS y RUL bajo sin empeorar dangerous_20.
- Riesgos metodologicos: el guard es una regla post-smoothing sobre predicciones internas; todavia no se actualizan conclusiones ni archivos globales finales.
- Confirmacion explicita: no se uso test final.
- Confirmacion explicita: no se usaron prefijos futuros; `no_future_prefixes_used=True`.
